# My Greatest Sin — Brian2CUDA v783 corridor smoke test

This notebook runs a **checksum-verified, bounded v783 corridor** on an NVIDIA Colab runtime. It is not a full-FlyWire simulation, animal measurement, WebGPU benchmark, or FlyBody controller. Select **Runtime → Change runtime type → T4 GPU** before running. The notebook refuses CPU fallback.

In [ ]:
!nvidia-smi
!nvcc --version
!pip -q install 'Brian2Cuda==1.0b1' pandas==2.2.3 pyarrow==18.1.0
import brian2cuda
brian2cuda.example_run()
print('PASS: upstream Brian2CUDA NVIDIA smoke test; this is not yet a FlyWire result.')

## Upload the two small experiment files

Upload `run_v783_corridor_brian2cuda.py`, `brian2cuda_corridor_fixture_smoke.py` and one signed corridor JSON, such as the locally generated `flywire-v783-sugar-mn9-corridor-v1/corridor.json`. Raw FlyWire packs must not be uploaded to this notebook.

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', sorted(uploaded))

In [ ]:
import hashlib

CORRIDOR_FILENAME = 'corridor.json'  # Replace with the exact uploaded filename.
EXPECTED_CORRIDOR_SHA256 = 'PASTE_64_HEX_SHA256_FROM_YOUR_CORRIDOR_MANIFEST'

if len(EXPECTED_CORRIDOR_SHA256) != 64 or any(char not in '0123456789abcdefABCDEF' for char in EXPECTED_CORRIDOR_SHA256):
    raise ValueError('Set EXPECTED_CORRIDOR_SHA256 before any model run.')
with open(CORRIDOR_FILENAME, 'rb') as handle:
    actual_sha256 = hashlib.sha256(handle.read()).hexdigest()
assert actual_sha256 == EXPECTED_CORRIDOR_SHA256.lower(), (actual_sha256, EXPECTED_CORRIDOR_SHA256)
print('Verified corridor SHA-256:', actual_sha256)
!python run_v783_corridor_brian2cuda.py --preflight
!python brian2cuda_corridor_fixture_smoke.py
!python run_v783_corridor_brian2cuda.py --corridor "{CORRIDOR_FILENAME}" --build-dir /content/v783-cuda-build --rate-hz 100 --expected-corridor-sha256 "{EXPECTED_CORRIDOR_SHA256}"
!cat /content/v783-cuda-build/run-report.json

A valid report includes the corridor SHA-256, rate, retained node/edge counts, hard caps and raw MN9 spike count. Record the Colab GPU name, driver, CUDA version, exact package versions, seed/protocol and report checksum before comparing it with the browser or offline runner. A successful fixture proves only CUDA compilation/execution, never FlyWire biology.